# Meter Maintenance Notebook

Use the main notebook for the normal pipeline run.

This notebook is for:
- inspecting one meter or a few meters closely
- comparing raw data to the current broken meter source entry
- reviewing raw readings captured inside remove/broken windows
- deciding whether a meter is still broken, repaired, misclassified, or needs a date change
- keeping a small update log
- creating a reviewed copy of the broken meter source file
- optionally writing confirmed changes back to the source file

### Files explanation

Files input:
- `running_list_broken_meters.csv` = **source** of broken meter file
- `removed_special_meter_data.csv` = **raw readings captured** inside remove/broken windows
- `special_meter_candidates.csv` = current review candidate workbook
- `special_meters_corrections_master_sheet.csv` = generated master corrections workbook

Files output:
- `broken_meter_update_log.csv` = maintenance update log
- `running_list_broken_meters_reviewed_copy.csv` = reviewed copy created from this notebook

In [ ]:
# TODO: make sure removed special is in order of meter, datetime
# rename interval to clean 
# TODO: add date parsing function from data_clean_TEST.py to this notebook
# TODO: add __file__
#TODO: change to load from server

# TODO: add restart marker to pdfs

### 1. Parameters

In [ ]:
############ CHANGE PARAMETERS AS NEEDED #############

# review every meter from the broken source file that has a status in BROKEN_STATUS_VALUES
review_all_broken_status_meters = True

# review_all_removed_meters = True, review all unique meters in removed_special_meter_data_file

# leave empty if using automatic broken meter selection
meters_to_review = []

# optional zoom window for plots
zoom_start = None  # e.g. "2025-08-01 00:00:00"
zoom_end = None     # e.g. "2025-10-01 00:00:00"

# if True, refresh the update-log rows for the selected meters
refresh_update_log_for_selected_meters = True

# whether to save changes back to the source broken meter file
# keep False unless ready to overwrite the source file
write_changes_to_source_file = False

######################################################

In [ ]:
# Data Directories
input_dir = "../data/extracts/"
output_dir = "../data/outputs/"
plot_dir = "../data/outputs/plots/"
maintenance_dir = "../data/outputs/maintenance/"


# TODO: change to server paths if running on server, make be all of its meter data or a subset
# Main raw meter data for this run
var_file = input_dir + "harvest_kwh_15min_" + "250723-260508.csv" #"250723-251017.csv"

# Main pipeline source/review files
meter_issues_candidates_file = input_dir + "special_meter_candidates.csv"
broken_meters_file = input_dir + "running_list_broken_meters.csv"

# Main pipeline generated files
meter_corrections_file = output_dir + "special_meters_corrections_master_sheet.csv"
removed_special_meter_data_file = output_dir + "removed_special_meter_data.csv"

# Maintenance files
broken_meter_update_log_file = maintenance_dir + "broken_meter_update_log.csv"
broken_meter_reviewed_copy_file = maintenance_dir + "running_list_broken_meters_reviewed_copy.csv"
maintenance_plot_file = plot_dir + "maintenance_review_selected_meters.pdf"


section ideas:

- load one meter or selected meters
- plot raw data around chosen dates
- compare against current broken list entry
- decide update: broken / repaired / not actually broken
- save changes back to the source file / or database


### 2. Imports

In [7]:
import os
import importlib
import numpy as np
import pandas as pd
import data_clean_TEST as dc
importlib.reload(dc)

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

### 3. Load Data

### 4. Data Processing and building review list

In [ ]:
# Load raw interval data
raw_df = pd.read_csv(var_file)
raw_df["datetime"] = pd.to_datetime(raw_df["datetime"])
raw_df = raw_df.dropna(subset=["datetime"]).copy()

# Pivot to wide meter dataframe with every meter as one column
pivoted_df = raw_df.pivot(index="datetime", columns="meter_name", values="meter_reading").reset_index()

# Fill missing timestamps to ensure continuous time series
full_df = dc.fill_missing_timestamps(pivoted_df, "15min")
data = full_df.set_index("datetime").sort_index()

######################################################

# Load broken meter source file
broken_source_df = dc.load_broken_meter_workbook(broken_meters_file)

# Build review meter list automatically from broken source statuses
data_cols_norm = {str(col).strip().lower() for col in data.columns}

all_broken_status_meters = sorted(
    broken_source_df["meter_name"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

meters_not_in_data = sorted([m for m in all_broken_status_meters if m not in data_cols_norm])

if review_all_broken_status_meters:
    meters_to_review = sorted([m for m in all_broken_status_meters if m in data_cols_norm])

print("all broken-status meters from source file:", len(all_broken_status_meters))
print("broken-status meters not found in raw data:", len(meters_not_in_data))

if meters_not_in_data:
    print("not in raw data:")
    print(meters_not_in_data)

######################################################

# Load candidate workbook
candidate_df = dc._load_existing_candidates(meter_issues_candidates_file)

######################################################

# Load generated master corrections file
master_corrections_df = pd.read_csv(meter_corrections_file)

######################################################

# Load removed raw data captured inside remove/broken windows
removed_df = pd.read_csv(removed_special_meter_data_file)
for col in ["datetime", "correction_start", "correction_end"]:
    if col in removed_df.columns:
        removed_df[col] = pd.to_datetime(removed_df[col], errors="coerce")

######################################################

print("\n" + "raw_df shape:", raw_df.shape)
print("data shape:", data.shape)
print("broken_source_df rows:", len(broken_source_df))
print("removed_df rows:", len(removed_df))


all broken-status meters from source file: 34
broken-status meters not found in raw data: 20
not in raw data:
['campus_center_ac_main', 'ching_complex_main', 'hale_noelani_all_towers_main', 'hale_noelani_tower_a_b', 'hale_noelani_tower_b', 'hale_noelani_tower_c', 'hale_noelani_tower_c_d', 'hale_noelani_tower_e', 'hig_substation_1_main', 'hig_substation_2_main', 'hig_substation_3_main', 'it_center_main', 'korean_studies_main', 'law_clinic_main', 'law_school_ac_main', 'pope_lab_main', 'quad_chiller_plant_main', 'saunders_hall_main_a', 'sinclair_lib_main', 'softball_tennis_main']

raw_df shape: (1499079, 3)
data shape: (27753, 95)
broken_source_df rows: 35
removed_df rows: 65479


### Meter review plots

#### Plot legend
- blue line = raw meter readings
- orange dots = raw readings captured in `removed_special_meter_data.csv`
- red spans = broken intervals from `running_list_broken_meters.csv`
- blue spans = candidate issue intervals from `special_meter_candidates.csv`
- purple spans = visual overlap of red and blue spans

In [ ]:
def plot_meter_maintenance_review(
    meter_name,
    data,
    broken_source_df,
    candidate_df,
    removed_df,
    zoom_start=None,
    zoom_end=None,
):
    meter_name_norm = str(meter_name).strip().lower()

    if meter_name_norm not in [str(col).strip().lower() for col in data.columns]:
        print(f"{meter_name} not found in data columns.")
        return None

    # map normalized meter name back to actual data column name
    data_col = None
    for col in data.columns:
        if str(col).strip().lower() == meter_name_norm:
            data_col = col
            break

    meter_series = data[data_col].copy()

    meter_broken = broken_source_df[broken_source_df["meter_name"] == meter_name_norm].copy()
    meter_candidates = candidate_df[candidate_df["meter_name"] == meter_name_norm].copy()
    meter_removed = removed_df[removed_df["meter_name"].astype(str).str.strip().str.lower() == meter_name_norm].copy()

    plot_start = pd.to_datetime(meter_series.index.min())
    plot_end = pd.to_datetime(meter_series.index.max())

    if zoom_start is not None:
        zoom_start_dt = pd.to_datetime(zoom_start)
        plot_start = max(plot_start, zoom_start_dt)
    if zoom_end is not None:
        zoom_end_dt = pd.to_datetime(zoom_end)
        plot_end = min(plot_end, zoom_end_dt)

    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(meter_series.index, meter_series.values, linewidth=1.2)
    ax.set_xlim(plot_start, plot_end)
    ax.set_title(meter_name_norm)
    ax.set_xlabel("Datetime")
    ax.set_ylabel("kWh")
    ax.grid(True, alpha=0.3)

    # broken source intervals = red spans
    for _, row in meter_broken.iterrows():
        start = plot_start if pd.isna(row["start_date"]) else max(pd.to_datetime(row["start_date"]), plot_start)
        end = plot_end if pd.isna(row["end_date"]) else min(pd.to_datetime(row["end_date"]), plot_end)
        if start <= end:
            ax.axvspan(start, end, alpha=0.2, color="red")

    # candidate intervals = blue spans
    for _, row in meter_candidates.iterrows():
        start = pd.to_datetime(row["start_datetime"], errors="coerce")
        end = pd.to_datetime(row["end_datetime"], errors="coerce")
        if pd.notna(start) and pd.notna(end):
            start = max(start, plot_start)
            end = min(end, plot_end)
            if start <= end:
                ax.axvspan(start, end, alpha=0.2, color="blue")

    # removed raw readings = orange points
    if not meter_removed.empty and "datetime" in meter_removed.columns and "meter_reading" in meter_removed.columns:
        removed_plot = meter_removed[
            (meter_removed["datetime"] >= plot_start) & (meter_removed["datetime"] <= plot_end)
        ].copy()
        if not removed_plot.empty:
            ax.scatter(
                removed_plot["datetime"],
                removed_plot["meter_reading"],
                s=10,
                alpha=0.8,
                marker="o",
                color="orange",
            )

    issue_types = [
        str(value).strip()
        for value in pd.unique(meter_candidates["issue_type"].dropna())
        if str(value).strip() != ""
    ]

    r2_values = [
        str(value).strip()
        for value in pd.unique(meter_candidates["r2"].dropna())
        if str(value).strip() != ""
    ]

    status_values = [
        str(value).strip()
        for value in pd.unique(meter_broken["status"].dropna())
        if str(value).strip() != ""
    ]

    annotation_lines = []
    if status_values:
        annotation_lines.append("Broken status: " + ", ".join(status_values))
    if issue_types:
        annotation_lines.append("Candidate issue type: " + ", ".join(issue_types))
    if not meter_removed.empty:
        annotation_lines.append(f"Removed raw points: {len(meter_removed)}")

    if annotation_lines:
        ax.text(
            0.05, 0.95,
            "\n".join(annotation_lines),
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.7)
        )

    plt.tight_layout()
    return fig


In [ ]:
# Plot selected meters
if not meters_to_review:
    print("Add one or more meter names to meters_to_review first.")
else:
    saved_count = 0
    with PdfPages(maintenance_plot_file) as pdf:
        for meter_name in meters_to_review:
            fig = plot_meter_maintenance_review(
                meter_name=meter_name,
                data=data,
                broken_source_df=broken_source_df,
                candidate_df=candidate_df,
                removed_df=removed_df,
                zoom_start=zoom_start,
                zoom_end=zoom_end,
            )
            if fig is not None:
                pdf.savefig(fig, bbox_inches="tight")
                plt.show()
                plt.close(fig)
                saved_count += 1

    if saved_count > 0:
        print(f"Saved maintenance review plots to {maintenance_plot_file}")


### Create or Refresh Update Log

`broken_meter_update_log.csv` column guide:
- `source_...` = what the broken meter file says right now
- `new_...` = what you think it should be changed to
- `action` = what kind of change you want to make
- `reason` = why you want to change it
- `approved` = whether you are ready to apply that change

In [ ]:
# update log helper functions:

def _dt_to_text(value):
    if pd.isna(value) or value == "":
        return ""
    dt = pd.to_datetime(value, errors="coerce")
    if pd.isna(dt):
        return ""
    return dt.strftime("%Y-%m-%d %H:%M:%S")

def _build_update_key(df):
    temp = df.copy()
    for col in ["meter_name", "source_start_date", "source_end_date", "source_status"]:
        if col not in temp.columns:
            temp[col] = ""
        temp[col] = temp[col].fillna("").astype(str).str.strip().str.lower()
    return (
        temp["meter_name"] + "||"
        + temp["source_start_date"] + "||"
        + temp["source_end_date"] + "||"
        + temp["source_status"]
    )

In [18]:
update_log_cols = [
    "meter_name",
    "source_start_date",
    "source_end_date",
    "source_status",
    "source_description",
    "action",
    "new_start_date",
    "new_end_date",
    "new_status",
    "new_description",
    "reason",
    "approved",
]

update_log_df = pd.read_csv(broken_meter_update_log_file)

for col in update_log_cols:
    if col not in update_log_df.columns:
        update_log_df[col] = ""

if refresh_update_log_for_selected_meters and meters_to_review:
    selected_rows = []

    for meter_name in meters_to_review:
        meter_name_norm = str(meter_name).strip().lower()
        meter_source = broken_source_df[broken_source_df["meter_name"] == meter_name_norm].copy()

        if meter_source.empty:
            selected_rows.append({
                "meter_name": meter_name_norm,
                "source_start_date": "",
                "source_end_date": "",
                "source_status": "",
                "source_description": "",
                "action": "",
                "new_start_date": "",
                "new_end_date": "",
                "new_status": "",
                "new_description": "",
                "reason": "",
                "approved": 0,
            })
        else:
            for _, row in meter_source.iterrows():
                selected_rows.append({
                    "meter_name": meter_name_norm,
                    "source_start_date": _dt_to_text(row["start_date"]),
                    "source_end_date": _dt_to_text(row["end_date"]),
                    "source_status": str(row["status"]).strip(),
                    "source_description": str(row["description"]).strip(),
                    "action": "",
                    "new_start_date": "",
                    "new_end_date": "",
                    "new_status": "",
                    "new_description": "",
                    "reason": "",
                    "approved": 0,
                })

    selected_log_df = pd.DataFrame(selected_rows, columns=update_log_cols)

    update_log_df["_key"] = _build_update_key(update_log_df)
    selected_log_df["_key"] = _build_update_key(selected_log_df)

    selected_keys = set(selected_log_df["_key"])

    preserved_existing = update_log_df[update_log_df["_key"].isin(selected_keys)].copy()
    preserved_other = update_log_df[~update_log_df["_key"].isin(selected_keys)].copy()

    combined_refresh_rows = pd.concat(
        [preserved_existing, selected_log_df],
        ignore_index=True,
        sort=False,
    )

    combined_refresh_rows = combined_refresh_rows.drop_duplicates(subset="_key", keep="first")
    update_log_df = pd.concat([preserved_other, combined_refresh_rows], ignore_index=True, sort=False)
    update_log_df = update_log_df.drop(columns="_key", errors="ignore")

update_log_df["approved"] = pd.to_numeric(update_log_df["approved"], errors="coerce").fillna(0).astype(int)
update_log_df = update_log_df[update_log_cols].copy()
update_log_df.to_csv(broken_meter_update_log_file, index=False)

display(
    update_log_df[
        update_log_df["meter_name"].isin([m.strip().lower() for m in meters_to_review])
    ] if meters_to_review else update_log_df.head(2)
)


,meter_name,source_start_date,source_end_date,source_status,source_description,action,new_start_date,new_end_date,new_status,new_description,reason,approved
0,ag_engineering_main,2025-02-19 00:00:00,NaN,broken,3/16/26 vernon says it needs to be reprogrammed,NaN,NaN,NaN,NaN,NaN,NaN,0
1,hper_klum_gym,NaN,NaN,broken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,malama_1_2_ehso_main,NaN,NaN,broken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,ag_engineering_mcc,2026-03-20 00:00:00,NaN,broken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,ag_engineering_mcc,2025-02-19 00:00:00,2025-09-03 00:00:00,broken,repaired,NaN,NaN,NaN,NaN,NaN,NaN,0
5,gilmore_hall_main_b,2025-08-19 00:00:00,NaN,broken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
6,gilmore_hall_mcc,2025-08-19 00:00:00,NaN,broken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
7,hig_noaa,2026-03-22 00:00:00,NaN,broken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
8,hig_panel_pb,2026-03-22 00:00:00,NaN,broken,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
9,music_complex_main,2026-03-13 00:00:00,2026-04-05 00:00:00,repaired,repaired over weekend,NaN,NaN,NaN,NaN,NaN,NaN,0


### Review Update log 

In [ ]:
if not write_changes_to_source_file:
    raise SystemExit("Set write_changes_to_source_file = True to apply changes to the source broken meter file.")